# Pipeline end to end


Esta es la plantilla más reutilizable para ejercicios tabulares completos.

Sirve cuando tienes un CSV con columnas numéricas/categóricas, valores faltantes y una target de clasificación. Si el problema es regresión, cambias el modelo final y las métricas.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "target"  # CAMBIAR

df = pd.read_csv("data.csv")

print("Forma:", df.shape)
print(df.head())
print("\nFaltantes:")
print(df.isna().sum().sort_values(ascending=False).head(10))
print("\nTarget:")
print(df[TARGET].value_counts(dropna=False))

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None,
)


In [ ]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))


## Para convertirla a regresión

Cambia:

```python
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
```

Y el modelo:

```python
("regressor", Ridge(alpha=1.0))
```

Luego evalúas con `MAE`, `RMSE` y `R2`.


## Por qué esta plantilla es segura

- El split ocurre antes del preprocesado.
- El `Pipeline` evita leakage.
- Las numéricas se imputan y escalan.
- Las categóricas se imputan y codifican.
- `handle_unknown="ignore"` evita romper si aparece una categoría nueva en test.
